# Шаг 15. Проверка дат

Поскольку в датасете нет полных дат (в формате YYYY-MM-DD), а есть только годы (release_year, years_from, years_to), будет проведена только валидация целочисленных годов на соответствие логике и историческим рамкам.

In [3]:
import pandas as pd
import numpy as np

# Загружаем очищенный датасет
df = pd.read_csv('df_consolidated.csv', dtype={
    'release_year': 'Int64',
    'aggregate_rating': 'Int64',
    'num_figures': 'Int64',
    'decade': 'Int64',
    'years_since_release': 'Int64'
})

CURRENT_YEAR = 2026  # Фиксируем текущий год для проверок

print("=== 1. Проверка года выпуска (release_year) ===")
# Игнорируем пропуски при расчете мин/макс
valid_release_years = df['release_year'].dropna()

print(f"Минимальный год выпуска: {valid_release_years.min()}")
print(f"Максимальный год выпуска: {valid_release_years.max()}")

# Проверка на даты из будущего
future_releases = df[df['release_year'] > CURRENT_YEAR]
print(f"Наборов с годом выпуска из будущего (> {CURRENT_YEAR}): {len(future_releases)}")

# Проверка на слишком старые даты (массовое производство пластиковых солдатиков началось ~1950-х)
# Airfix, например, начал массовый выпуск в 1958 году.
too_old_releases = df[df['release_year'] < 1950]
print(f"Наборов с годом выпуска раньше 1950: {len(too_old_releases)}")
if len(too_old_releases) > 0:
    print("Примеры 'слишком старых' наборов:")
    print(too_old_releases[['id', 'header', 'release_year']].head())

print("\n=== 2. Проверка исторических периодов (years_from, years_to) ===")
valid_from = df['years_from'].dropna()
valid_to = df['years_to'].dropna()

print(f"Минимальный год исторического периода (years_from): {valid_from.min()}")
print(f"Максимальный год исторического периода (years_to): {valid_to.max()}")

# Логическая проверка: год начала не может быть больше года окончания
invalid_periods = df[df['years_from'] > df['years_to']]
print(f"Наборов, где год начала больше года окончания: {len(invalid_periods)}")
if len(invalid_periods) > 0:
    print("Примеры некорректных периодов:")
    print(invalid_periods[['id', 'header', 'years_from', 'years_to']].head())

print("\n=== Итоговый вывод по датам ===")
if len(future_releases) == 0 and len(invalid_periods) == 0:
    print("✅ Временные рамки данных корректны и не содержат критических аномалий.")
else:
    print("⚠️ Обнаружены аномалии во временных данных. Рекомендуется ручная проверка.")

=== 1. Проверка года выпуска (release_year) ===
Минимальный год выпуска: 1958
Максимальный год выпуска: 2026
Наборов с годом выпуска из будущего (> 2026): 0
Наборов с годом выпуска раньше 1950: 0

=== 2. Проверка исторических периодов (years_from, years_to) ===
Минимальный год исторического периода (years_from): -2024
Максимальный год исторического периода (years_to): 2024
Наборов, где год начала больше года окончания: 0

=== Итоговый вывод по датам ===
✅ Временные рамки данных корректны и не содержат критических аномалий.
